import os
import pickle
import mediapipe as mp
import cv2

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    min_detection_confidence=0.3,
    max_num_hands=2
)

DATA_DIR = './data'

data = []
labels = []

def extract_hand_features(hand_landmarks):
    x_ = []
    y_ = []

    for lm in hand_landmarks.landmark:
        x_.append(lm.x)
        y_.append(lm.y)

    data_aux = []

    for lm in hand_landmarks.landmark:
        data_aux.append(lm.x - min(x_))
        data_aux.append(lm.y - min(y_))

    return data_aux  # 42 features


for dir_ in os.listdir(DATA_DIR):
    for img_path in os.listdir(os.path.join(DATA_DIR, dir_)):

        img = cv2.imread(os.path.join(DATA_DIR, dir_, img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        results = hands.process(img_rgb)

        # Initialize padded hands
        left_hand = [0] * 42
        right_hand = [0] * 42

        if results.multi_hand_landmarks and results.multi_handedness:

            for i, hand_landmarks in enumerate(results.multi_hand_landmarks):

                label = results.multi_handedness[i].classification[0].label

                features = extract_hand_features(hand_landmarks)

                if label == 'Left':
                    left_hand = features
                elif label == 'Right':
                    right_hand = features

        final_data = left_hand + right_hand  # 84 features

        data.append(final_data)
        labels.append(dir_)

with open('data.pickle', 'wb') as f:
    pickle.dump({'data': data, 'labels': labels}, f)

In [3]:
import os
import pickle
import mediapipe as mp
import cv2

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    min_detection_confidence=0.3,
    max_num_hands=1  # single hand only
)

DATA_DIR = './data'

data = []
labels = []

def extract_hand_features(hand_landmarks):
    x_ = []
    y_ = []

    for lm in hand_landmarks.landmark:
        x_.append(lm.x)
        y_.append(lm.y)

    data_aux = []

    for lm in hand_landmarks.landmark:
        data_aux.append(lm.x - min(x_))
        data_aux.append(lm.y - min(y_))

    return data_aux  # 42 features


for dir_ in os.listdir(DATA_DIR):
    for img_path in os.listdir(os.path.join(DATA_DIR, dir_)):

        img = cv2.imread(os.path.join(DATA_DIR, dir_, img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        results = hands.process(img_rgb)

        # ❌ Skip if no hand detected
        if not results.multi_hand_landmarks:
            continue

        hand_landmarks = results.multi_hand_landmarks[0]

        features = extract_hand_features(hand_landmarks)

        data.append(features)
        labels.append(dir_)

with open('data.pickle', 'wb') as f:
    pickle.dump({'data': data, 'labels': labels}, f)

print("✅ Dataset created successfully")
print(f"Total samples: {len(data)}")

✅ Dataset created successfully
Total samples: 7400
